# Marketing Analytics — Data Cleaning

I'm using two datasets for this project: a customer-level marketing campaign dataset (2,240
customers, their spend, and how they responded to 5 past campaigns) and a PPC performance
dataset (1,000 paid ad campaigns across platforms). Before either one goes into Tableau, I
wanted to actually audit them rather than just clean whatever pandas flagged automatically —
so this notebook walks through what I found and why I made the calls I made.

Both cleaned files get exported to `data/cleaned/` and connected directly in Tableau as two
separate dashboard tabs, since they don't share a join key.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

---
## Part 1 — Marketing Campaign (customer-level data)

### 1.1 Load and get oriented

In [2]:
df1 = pd.read_excel("data/raw/marketing_campaign.xlsx")
print("Shape:", df1.shape)
df1.head()

Shape: (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


In [3]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

### 1.2 Data quality audit

In [4]:
# Missing values
df1.isnull().sum()[df1.isnull().sum() > 0]

Income    24
dtype: int64

In [5]:
# Duplicate customer IDs
print("Duplicate IDs:", df1['ID'].duplicated().sum())

Duplicate IDs: 0


In [6]:
# Categorical sanity check — Marital_Status
df1['Marital_Status'].value_counts()

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

First thing I checked was the categorical fields, since those are the easiest to miss
if you only look at `.isnull()`. `Marital_Status` turned out to have a couple of joke
entries — `Absurd` and `YOLO` — plus `Alone`, which is really just `Single` worded
differently. None of these are real segments, so I'm folding them in rather than
treating them as their own category.

In [7]:
# Outlier check — birth year (implies unrealistic age)
df1[df1['Year_Birth'] < 1930][['ID', 'Year_Birth', 'Education', 'Marital_Status']]

,ID,Year_Birth,Education,Marital_Status
192,7829,1900,2n Cycle,Divorced
239,11004,1893,2n Cycle,Single
339,1150,1899,PhD,Together


In [8]:
# Outlier check — income
df1[df1['Income'] > 200000][['ID', 'Income']]

,ID,Income
2233,9432,666666.0


Next I checked for numeric outliers that `.describe()` alone doesn't always surface
clearly. Three customers have birth years in the 1890s, which would make them over
110 years old — and one customer's income is $666,666, about 9x the 75th percentile.
I'm treating both as data-entry errors rather than trying to impute or cap them,
since inventing a "corrected" age or income would be worse than just dropping 4 rows
out of 2,240.

### 1.3 Clean

In [9]:
df1_clean = df1.copy()

# 1. Parse date
df1_clean['Dt_Customer'] = pd.to_datetime(df1_clean['Dt_Customer'])

# 2. Fix Marital_Status categories
marital_map = {
    'Alone': 'Single',
    'Absurd': 'Other',
    'YOLO': 'Other'
}
df1_clean['Marital_Status'] = df1_clean['Marital_Status'].replace(marital_map)

# 3. Drop unrealistic birth years and the income outlier
df1_clean = df1_clean[df1_clean['Year_Birth'] >= 1930]
df1_clean = df1_clean[df1_clean['Income'] < 200000]

# 4. Impute missing Income with the median for that customer's Education level
df1_clean['Income'] = df1_clean.groupby('Education')['Income'].transform(
    lambda x: x.fillna(x.median())
)

print("Rows remaining:", len(df1_clean))
print("Remaining nulls:", df1_clean.isnull().sum().sum())

Rows remaining: 2212
Remaining nulls: 0


In [10]:
# Verify the fix
df1_clean['Marital_Status'].value_counts()

Marital_Status
Married     857
Together    571
Single      473
Divorced    231
Widow        76
Other         4
Name: count, dtype: int64

In [11]:
# Age and tenure
df1_clean['Age'] = 2014 - df1_clean['Year_Birth']
snapshot_date = df1_clean['Dt_Customer'].max() + pd.Timedelta(days=1)
df1_clean['Customer_Tenure_Days'] = (snapshot_date - df1_clean['Dt_Customer']).dt.days

# Spend and purchases
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df1_clean['Total_Spend'] = df1_clean[mnt_cols].sum(axis=1)

purchase_cols = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
df1_clean['Total_Purchases'] = df1_clean[purchase_cols].sum(axis=1)

# Household
df1_clean['Total_Kids'] = df1_clean['Kidhome'] + df1_clean['Teenhome']

# Campaign response
cmp_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
df1_clean['Campaigns_Accepted'] = df1_clean[cmp_cols].sum(axis=1)
df1_clean['Campaign_Response_Rate'] = df1_clean['Campaigns_Accepted'] / len(cmp_cols)

# Income band
df1_clean['Income_Band'] = pd.cut(
    df1_clean['Income'],
    bins=[0, 30000, 60000, 90000, np.inf],
    labels=['Low (<30k)', 'Mid (30-60k)', 'High (60-90k)', 'Very High (90k+)']
)

# Deal dependency (guard against divide-by-zero for customers with 0 purchases)
df1_clean['Deal_Dependency_Rate'] = np.where(
    df1_clean['Total_Purchases'] > 0,
    df1_clean['NumDealsPurchases'] / df1_clean['Total_Purchases'],
    0
)

df1_clean[['Age', 'Customer_Tenure_Days', 'Total_Spend', 'Total_Purchases',
           'Campaign_Response_Rate', 'Income_Band', 'Deal_Dependency_Rate']].describe(include='all')

,Age,Customer_Tenure_Days,Total_Spend,Total_Purchases,Campaign_Response_Rate,Income_Band,Deal_Dependency_Rate
count,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212,2212.000000
unique,NaN,NaN,NaN,NaN,NaN,4,NaN
top,NaN,NaN,NaN,NaN,NaN,Mid (30-60k),NaN
freq,NaN,NaN,NaN,NaN,NaN,1004,NaN
mean,45.086347,354.714286,607.268083,14.891501,0.074819,NaN,0.180195
std,11.701599,202.494886,602.513364,7.671629,0.148836,NaN,0.111204
min,18.000000,1.000000,5.000000,0.000000,0.000000,NaN,0.000000
25%,37.000000,181.000000,69.000000,8.000000,0.000000,NaN,0.076923
50%,44.000000,357.000000,397.000000,15.000000,0.000000,NaN,0.166667
75%,55.000000,530.000000,1048.000000,21.000000,0.166667,NaN,0.250000


### 1.4 Export

In [12]:
output_cols = list(df1.columns) + [
    'Age', 'Customer_Tenure_Days', 'Total_Spend', 'Total_Purchases',
    'Total_Kids', 'Campaigns_Accepted', 'Campaign_Response_Rate',
    'Income_Band', 'Deal_Dependency_Rate'
]
df1_clean[output_cols].to_csv("data/cleaned/marketing_campaign_cleaned.csv", index=False)
print("Saved marketing_campaign_cleaned.csv —", df1_clean.shape[0], "rows,", len(output_cols), "columns")

Saved marketing_campaign_cleaned.csv — 2212 rows, 38 columns


---
## Part 2 — PPC Campaign Performance (ad-level data)

### 2.1 Load and get oriented

In [13]:
df2 = pd.read_excel("data/raw/ppc_campaign_performance_data.xlsx")
print("Shape:", df2.shape)
df2.head()

Shape: (1000, 19)


,Campaign_ID,Budget,Clicks,CTR,CPC,Conversions,CPA,Conversion_Rate,Duration,Platform,Content_Type,Target_Age,Target_Gender,Region,Revenue,Spend,ROAS,Date,Impressions
0,C3578,6390,401,0.0461,15.94,174,36.72,0.4339,20,Instagram,Carousel,35-44,Male,Europe,27840,6453.9,4.31,2025-01-19,8698
1,C6702,9870,1286,0.2860,7.67,821,12.02,0.6384,28,LinkedIn,Text,55+,Male,Africa,128076,10067.4,12.72,2025-01-22,4496
2,C9725,7700,1684,0.2122,4.57,1060,7.26,0.6295,15,Instagram,Video,35-44,Other,North America,193980,7623.0,25.45,2024-07-23,7935
3,C9472,8420,444,0.0961,18.96,308,27.34,0.6937,25,Google,Text,25-34,Male,North America,24024,8504.2,2.82,2024-04-20,4620
4,C7601,8470,1912,0.3652,4.43,1428,5.93,0.7469,9,Google,Text,25-34,Other,Europe,277032,8046.5,34.43,2024-08-07,5235


In [14]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Campaign_ID      1000 non-null   str    
 1   Budget           1000 non-null   int64  
 2   Clicks           1000 non-null   int64  
 3   CTR              1000 non-null   float64
 4   CPC              1000 non-null   float64
 5   Conversions      1000 non-null   int64  
 6   CPA              1000 non-null   float64
 7   Conversion_Rate  1000 non-null   float64
 8   Duration         1000 non-null   int64  
 9   Platform         1000 non-null   str    
 10  Content_Type     1000 non-null   str    
 11  Target_Age       1000 non-null   str    
 12  Target_Gender    1000 non-null   str    
 13  Region           1000 non-null   str    
 14  Revenue          1000 non-null   int64  
 15  Spend            1000 non-null   float64
 16  ROAS             1000 non-null   float64
 17  Date             1000 non-

### 2.2 Data quality audit

In [15]:
print("Missing values:", df2.isnull().sum().sum())
print("Duplicate Campaign_IDs:", df2['Campaign_ID'].duplicated().sum())
print("Fully duplicated rows:", df2.duplicated().sum())

Missing values: 0
Duplicate Campaign_IDs: 46
Fully duplicated rows: 0


In [16]:
# Look at a few duplicated Campaign_IDs
df2[df2['Campaign_ID'].duplicated(keep=False)].sort_values('Campaign_ID').head(6)[
    ['Campaign_ID', 'Platform', 'Date', 'Clicks', 'Impressions']
]

,Campaign_ID,Platform,Date,Clicks,Impressions
695,C1179,Google,2024-11-19,642,8891
744,C1179,LinkedIn,2025-01-06,1831,4587
439,C2099,Instagram,2024-05-16,765,5158
787,C2099,Facebook,2024-12-22,1498,3064
889,C2258,Google,2025-01-12,1936,4597
572,C2258,Facebook,2024-02-10,82,9534


In [17]:
# Check whether the pre-calculated efficiency metrics were built on actual Spend or planned Budget
df2['CPC_from_spend'] = (df2['Spend'] / df2['Clicks']).round(2)
df2['CPC_from_budget'] = (df2['Budget'] / df2['Clicks']).round(2)

print("CPC matches Spend-based calc:", (df2['CPC_from_spend'] == df2['CPC']).mean())
print("CPC matches Budget-based calc:", (df2['CPC_from_budget'] == df2['CPC']).mean())

CPC matches Spend-based calc: 0.089
CPC matches Budget-based calc: 1.0


This is the one that actually surprised me. I wanted to sanity-check the pre-built
`CPC` and `CPA` columns before trusting them for the dashboard, so I recalculated
both manually and compared. Turns out they were built against `Budget` — the planned
spend — not `Spend`, the actual amount charged. That's a real problem for a dashboard
meant to show ad efficiency: a campaign that blew past its budget would show up
looking *more* efficient than it actually was, just because of which number sits in
the denominator.

I recalculated `CPC_Actual`, `CPA_Actual`, and `ROAS_Actual` against real `Spend`,
and kept the original Budget-based columns alongside them so the difference is
visible rather than silently overwritten.

### 2.3 Clean

In [18]:
df2_clean = df2.drop(columns=['CPC_from_spend', 'CPC_from_budget']).copy()

# Unique row ID
df2_clean.insert(0, 'Campaign_Row_ID', range(1, len(df2_clean) + 1))

# Parse date, extract parts
df2_clean['Date'] = pd.to_datetime(df2_clean['Date'])
df2_clean['Month'] = df2_clean['Date'].dt.to_period('M').astype(str)
df2_clean['Quarter'] = df2_clean['Date'].dt.to_period('Q').astype(str)

# Recalculate efficiency metrics against actual Spend
df2_clean['CPC_Actual'] = (df2_clean['Spend'] / df2_clean['Clicks']).round(2)
df2_clean['CPA_Actual'] = np.where(
    df2_clean['Conversions'] > 0,
    (df2_clean['Spend'] / df2_clean['Conversions']).round(2),
    np.nan
)
df2_clean['ROAS_Actual'] = (df2_clean['Revenue'] / df2_clean['Spend']).round(2)

# Budget pacing
df2_clean['Budget_Variance'] = (df2_clean['Spend'] - df2_clean['Budget']).round(2)
df2_clean['Budget_Variance_Pct'] = (df2_clean['Budget_Variance'] / df2_clean['Budget']).round(4)

df2_clean[['Platform', 'Spend', 'Budget', 'CPC', 'CPC_Actual', 'CPA', 'CPA_Actual',
           'ROAS', 'ROAS_Actual', 'Budget_Variance_Pct']].head(8)

,Platform,Spend,Budget,CPC,CPC_Actual,CPA,CPA_Actual,ROAS,ROAS_Actual,Budget_Variance_Pct
0,Instagram,6453.9,6390,15.94,16.09,36.72,37.09,4.31,4.31,0.01
1,LinkedIn,10067.4,9870,7.67,7.83,12.02,12.26,12.72,12.72,0.02
2,Instagram,7623.0,7700,4.57,4.53,7.26,7.19,25.45,25.45,-0.01
3,Google,8504.2,8420,18.96,19.15,27.34,27.61,2.82,2.82,0.01
4,Google,8046.5,8470,4.43,4.21,5.93,5.63,34.43,34.43,-0.05
5,YouTube,5177.5,5450,8.22,7.81,9.01,8.56,4.21,4.21,-0.05
6,Instagram,4508.4,4420,4.23,4.32,9.78,9.97,11.43,11.43,0.02
7,YouTube,7334.0,7720,7.71,7.33,8.43,8.01,9.74,9.74,-0.05


In [19]:
df2_clean.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Campaign_Row_ID,1000.0,NaN,NaN,NaN,500.5,1.0,250.75,500.5,750.25,1000.0,288.819436
Campaign_ID,1000,954,C5910,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Budget,1000.0,NaN,NaN,NaN,5961.99,2030.0,4007.5,5855.0,7960.0,9990.0,2326.091753
Clicks,1000.0,NaN,NaN,NaN,1013.229,50.0,514.25,996.5,1491.5,1996.0,563.971223
CTR,1000.0,NaN,NaN,NaN,0.258417,0.0061,0.094275,0.18215,0.3069,1.7851,0.262449
CPC,1000.0,NaN,NaN,NaN,11.23073,1.08,3.7075,5.755,11.285,132.31,16.026811
Conversions,1000.0,NaN,NaN,NaN,505.672,10.0,153.75,396.5,772.25,1894.0,424.76226
CPA,1000.0,NaN,NaN,NaN,40.96617,1.67,7.1575,14.245,36.84,733.08,76.623075
Conversion_Rate,1000.0,NaN,NaN,NaN,0.508329,0.0096,0.267,0.51715,0.751675,1.0,0.284814
Duration,1000.0,NaN,NaN,NaN,18.313,7.0,12.0,18.0,24.0,30.0,6.952796


### 2.4 Export

In [20]:
df2_clean.to_csv("data/cleaned/ppc_campaign_performance_cleaned.csv", index=False)
print("Saved ppc_campaign_performance_cleaned.csv —", df2_clean.shape[0], "rows,", df2_clean.shape[1], "columns")

Saved ppc_campaign_performance_cleaned.csv — 1000 rows, 27 columns


## Where this leaves things

Marketing Campaign dataset went from 2,240 rows to 2,212 after dropping the age/income
outliers, plus 9 new columns for the KPIs I actually need in Tableau (spend, tenure,
campaign response rate, income band, etc.). PPC dataset stayed at 1,000 rows but picked
up a real unique ID and the corrected efficiency metrics.

Next: connect both cleaned CSVs in Tableau as separate tabs, build the KPI cards and
segment charts on the customer side, and channel/platform performance on the ad side.